# Notebook 02: Model Training & Evaluation
## Why Did My Model Predict That? — Heart Disease Explainability

**Goal**: Train Random Forest and XGBoost classifiers, evaluate performance with multiple metrics, and save models for SHAP/LIME analysis.

### Models Trained
- **Random Forest**: Ensemble of decision trees with bagging
- **XGBoost**: Gradient boosting with regularization

### Evaluation Metrics
- Accuracy, Precision, Recall, F1-Score, AUC-ROCimport numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                                                          confusion_matrix, roc_curve)
                                                          import xgboost as xgb
                                                          import warnings
                                                          warnings.filterwarnings('ignore')
                                                          
                                                          # Load preprocessed data
                                                          with open('/tmp/preprocessed_data.pkl', 'rb') as f:
                                                              data = pickle.load(f)
                                                              
                                                              X_train = data['X_train']
                                                              X_test  = data['X_test']
                                                              y_train = data['y_train']
                                                              y_test  = data['y_test']
                                                              feature_names = data['feature_names']
                                                              
                                                              print(f"✅ Data loaded | Train: {X_train.shape} | Test: {X_test.shape}")
                                                              print(f"Features: {feature_names}"))

In [0]:

xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                                use_label_encoder=False, eval_metric='logloss',
                                                                random_state=42)
                                                                xgb_model.fit(X_train, y_train)

                                                                # Evaluate both models
                                                                def evaluate_model(model, X_test, y_test, name):
                                                                    y_pred = model.predict(X_test)
                                                                        y_prob = model.predict_proba(X_test)[:, 1]
                                                                            return {
                                                                                    'Model': name,
                                                                                            'Accuracy': accuracy_score(y_test, y_pred),
                                                                                                    'Precision': precision_score(y_test, y_pred),
                                                                                                            'Recall': recall_score(y_test, y_pred),
                                                                                                                    'F1-Score': f1_score(y_test, y_pred),
                                                                                                                            'AUC-ROC': roc_auc_score(y_test, y_prob)
                                                                                                                                }

                                                                                                                                results = [
                                                                                                                                    evaluate_model(rf_model, X_test, y_test, 'Random Forest'),
                                                                                                                                        evaluate_model(xgb_model, X_test, y_test, 'XGBoost')
                                                                                                                                        ]
                                                                                                                                        results_df = pd.DataFrame(results).set_index('Model')
                                                                                                                                        print("=== Model Performance Comparison ===")
                                                                                                                                        display(results_df.round(4))

In [0]:
# ROC Curve and Confusion Matrix Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
for model, name, color in [(rf_model, 'Random Forest', 'steelblue'), 
                            (xgb_model, 'XGBoost', 'darkorange')]:
                                y_prob = model.predict_proba(X_test)[:, 1]
                                    fpr, tpr, _ = roc_curve(y_test, y_prob)
                                        auc = roc_auc_score(y_test, y_prob)
                                            axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)
                                            axes[0].plot([0,1],[0,1], 'k--', alpha=0.5, label='Random Classifier')
                                            axes[0].set_title('ROC Curves — Model Comparison', fontsize=13, fontweight='bold')
                                            axes[0].set_xlabel('False Positive Rate')
                                            axes[0].set_ylabel('True Positive Rate')
                                            axes[0].legend()

                                            # Confusion Matrix for best model (XGBoost)
                                            y_pred_xgb = xgb_model.predict(X_test)
                                            cm = confusion_matrix(y_test, y_pred_xgb)
                                            sns_cm = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                                                       xticklabels=['No Disease', 'Disease'],
                                                                  yticklabels=['No Disease', 'Disease'])
                                                                  axes[1].set_title('XGBoost Confusion Matrix', fontsize=13, fontweight='bold')
                                                                  axes[1].set_xlabel('Predicted')
                                                                  axes[1].set_ylabel('Actual')
                                                                  plt.tight_layout()
                                                                  plt.show()

                                                                  # Save models to file
                                                                  models_dict = {'rf_model': rf_model, 'xgb_model': xgb_model, 
                                                                                 'feature_names': feature_names}
                                                                                 with open('/tmp/models.pkl', 'wb') as f:
                                                                                     pickle.dump(models_dict, f)
                                                                                     print("✅ Models saved to /tmp/models.pkl")